In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df_train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

In [4]:
df_train_id = df_train['id']
df_test_id = df_test['id']

df_train.drop('id',axis=1,inplace=True)
df_test.drop('id',axis=1,inplace=True)

In [5]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Age                      630000 non-null  int64  
 1   Sex                      630000 non-null  int64  
 2   Chest pain type          630000 non-null  int64  
 3   BP                       630000 non-null  int64  
 4   Cholesterol              630000 non-null  int64  
 5   FBS over 120             630000 non-null  int64  
 6   EKG results              630000 non-null  int64  
 7   Max HR                   630000 non-null  int64  
 8   Exercise angina          630000 non-null  int64  
 9   ST depression            630000 non-null  float64
 10  Slope of ST              630000 non-null  int64  
 11  Number of vessels fluro  630000 non-null  int64  
 12  Thallium                 630000 non-null  int64  
 13  Heart Disease            630000 non-null  object 
dtypes: f

In [6]:
df_train.head()

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [7]:
categorical_cols = ['Sex','Chest pain type','FBS over 120','EKG results','Exercise angina',	'Slope of ST','Number of vessels fluro','Thallium']
numerical_cols = ['Age','BP','Cholesterol','Max HR','ST depression']

In [8]:
map_trget = {
    'Absence':0,
    'Presence':1
}

In [9]:
target = 'Heart Disease'
df_train[target] = df_train[target].map(map_trget)

target_df = df_train[target]
target_df

0         1
1         0
2         0
3         0
4         1
         ..
629995    0
629996    0
629997    1
629998    1
629999    0
Name: Heart Disease, Length: 630000, dtype: int64

In [10]:
train_categorical = df_train[categorical_cols]
train_numerical = df_train[numerical_cols]

test_categorical = df_test[categorical_cols]
test_numerical = df_test[numerical_cols]

In [11]:
train_categorical.head()

,Sex,Chest pain type,FBS over 120,EKG results,Exercise angina,Slope of ST,Number of vessels fluro,Thallium
0,1,4,0,0,1,2,2,7
1,1,1,0,2,0,1,0,3
2,0,2,0,2,0,1,0,3
3,0,3,0,2,0,2,0,3
4,1,4,0,2,1,2,3,3


In [12]:
test_categorical

,Sex,Chest pain type,FBS over 120,EKG results,Exercise angina,Slope of ST,Number of vessels fluro,Thallium
0,1,3,0,2,1,2,3,3
1,0,2,0,0,0,1,0,3
2,1,4,0,0,1,2,3,7
3,0,3,0,0,0,1,0,3
4,1,1,0,0,0,2,0,7
...,...,...,...,...,...,...,...,...
269995,1,2,0,0,0,1,0,7
269996,1,4,0,0,0,2,0,3
269997,1,3,1,0,0,1,0,3
269998,1,4,0,2,0,1,0,3


In [13]:
train_numerical

,Age,BP,Cholesterol,Max HR,ST depression
0,58,152,239,158,3.6
1,52,125,325,171,0.0
2,56,160,188,151,0.0
3,44,134,229,150,1.0
4,58,140,234,125,3.8
...,...,...,...,...,...
629995,56,110,226,132,0.0
629996,54,128,249,150,0.0
629997,67,130,275,149,0.0
629998,52,140,199,157,0.0


In [14]:
test_numerical

,Age,BP,Cholesterol,Max HR,ST depression
0,58,120,288,145,0.8
1,55,120,209,172,0.0
2,54,120,268,150,0.0
3,44,112,177,168,0.9
4,43,138,267,163,1.8
...,...,...,...,...,...
269995,58,120,222,172,1.0
269996,58,132,289,172,2.8
269997,63,108,201,158,0.8
269998,59,120,274,163,0.5


In [15]:
train_categorical.isna().sum()

Sex                        0
Chest pain type            0
FBS over 120               0
EKG results                0
Exercise angina            0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
dtype: int64

In [16]:
test_categorical.isna().sum()

Sex                        0
Chest pain type            0
FBS over 120               0
EKG results                0
Exercise angina            0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
dtype: int64

In [17]:
train_numerical.isnull().sum()

Age              0
BP               0
Cholesterol      0
Max HR           0
ST depression    0
dtype: int64

In [18]:
test_numerical.isnull().sum()

Age              0
BP               0
Cholesterol      0
Max HR           0
ST depression    0
dtype: int64

In [19]:
train_df_ohe = pd.get_dummies(train_categorical,columns=categorical_cols)
train_df_ohe.head()

,Sex_0,Sex_1,Chest pain type_1,Chest pain type_2,Chest pain type_3,Chest pain type_4,FBS over 120_0,FBS over 120_1,EKG results_0,EKG results_1,...,Slope of ST_1,Slope of ST_2,Slope of ST_3,Number of vessels fluro_0,Number of vessels fluro_1,Number of vessels fluro_2,Number of vessels fluro_3,Thallium_3,Thallium_6,Thallium_7
0,False,True,False,False,False,True,True,False,True,False,...,False,True,False,False,False,True,False,False,False,True
1,False,True,True,False,False,False,True,False,False,False,...,True,False,False,True,False,False,False,True,False,False
2,True,False,False,True,False,False,True,False,False,False,...,True,False,False,True,False,False,False,True,False,False
3,True,False,False,False,True,False,True,False,False,False,...,False,True,False,True,False,False,False,True,False,False
4,False,True,False,False,False,True,True,False,False,False,...,False,True,False,False,False,False,True,True,False,False


In [20]:
test_df_one = pd.get_dummies(test_categorical,columns=categorical_cols)
test_df_one.head()

,Sex_0,Sex_1,Chest pain type_1,Chest pain type_2,Chest pain type_3,Chest pain type_4,FBS over 120_0,FBS over 120_1,EKG results_0,EKG results_1,...,Slope of ST_1,Slope of ST_2,Slope of ST_3,Number of vessels fluro_0,Number of vessels fluro_1,Number of vessels fluro_2,Number of vessels fluro_3,Thallium_3,Thallium_6,Thallium_7
0,False,True,False,False,True,False,True,False,False,False,...,False,True,False,False,False,False,True,True,False,False
1,True,False,False,True,False,False,True,False,True,False,...,True,False,False,True,False,False,False,True,False,False
2,False,True,False,False,False,True,True,False,True,False,...,False,True,False,False,False,False,True,False,False,True
3,True,False,False,False,True,False,True,False,True,False,...,True,False,False,True,False,False,False,True,False,False
4,False,True,True,False,False,False,True,False,True,False,...,False,True,False,True,False,False,False,False,False,True


In [21]:
df_train = pd.concat([train_numerical,train_df_ohe],axis=1)
df_test = pd.concat([test_numerical,test_df_one],axis=1)

df_train.head()

,Age,BP,Cholesterol,Max HR,ST depression,Sex_0,Sex_1,Chest pain type_1,Chest pain type_2,Chest pain type_3,...,Slope of ST_1,Slope of ST_2,Slope of ST_3,Number of vessels fluro_0,Number of vessels fluro_1,Number of vessels fluro_2,Number of vessels fluro_3,Thallium_3,Thallium_6,Thallium_7
0,58,152,239,158,3.6,False,True,False,False,False,...,False,True,False,False,False,True,False,False,False,True
1,52,125,325,171,0.0,False,True,True,False,False,...,True,False,False,True,False,False,False,True,False,False
2,56,160,188,151,0.0,True,False,False,True,False,...,True,False,False,True,False,False,False,True,False,False
3,44,134,229,150,1.0,True,False,False,False,True,...,False,True,False,True,False,False,False,True,False,False
4,58,140,234,125,3.8,False,True,False,False,False,...,False,True,False,False,False,False,True,True,False,False


In [22]:
df_test.head()

,Age,BP,Cholesterol,Max HR,ST depression,Sex_0,Sex_1,Chest pain type_1,Chest pain type_2,Chest pain type_3,...,Slope of ST_1,Slope of ST_2,Slope of ST_3,Number of vessels fluro_0,Number of vessels fluro_1,Number of vessels fluro_2,Number of vessels fluro_3,Thallium_3,Thallium_6,Thallium_7
0,58,120,288,145,0.8,False,True,False,False,True,...,False,True,False,False,False,False,True,True,False,False
1,55,120,209,172,0.0,True,False,False,True,False,...,True,False,False,True,False,False,False,True,False,False
2,54,120,268,150,0.0,False,True,False,False,False,...,False,True,False,False,False,False,True,False,False,True
3,44,112,177,168,0.9,True,False,False,False,True,...,True,False,False,True,False,False,False,True,False,False
4,43,138,267,163,1.8,False,True,True,False,False,...,False,True,False,True,False,False,False,False,False,True


In [23]:
X = df_train
y = target_df

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [25]:
cat_cols = train_df_ohe.columns.values
cat_cols

array(['Sex_0', 'Sex_1', 'Chest pain type_1', 'Chest pain type_2',
       'Chest pain type_3', 'Chest pain type_4', 'FBS over 120_0',
       'FBS over 120_1', 'EKG results_0', 'EKG results_1',
       'EKG results_2', 'Exercise angina_0', 'Exercise angina_1',
       'Slope of ST_1', 'Slope of ST_2', 'Slope of ST_3',
       'Number of vessels fluro_0', 'Number of vessels fluro_1',
       'Number of vessels fluro_2', 'Number of vessels fluro_3',
       'Thallium_3', 'Thallium_6', 'Thallium_7'], dtype=object)

In [26]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42
)

lgbm = LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    verbose=-1
)

gb = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

models = {
    'XGBoost': xgb,
    'LightGBM': lgbm,
    'GradientBoosting': gb
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    print(f"{name}: ROC-AUC = {scores.mean():.5f} (+/- {scores.std():.5f})")

XGBoost: ROC-AUC = 0.95519 (+/- 0.00026)
LightGBM: ROC-AUC = 0.95522 (+/- 0.00024)
GradientBoosting: ROC-AUC = 0.95492 (+/- 0.00020)


In [28]:
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': 42
    }

    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"Best ROC-AUC: {study.best_value:.5f}")
print(f"Best params: {study.best_params}")

[I 2026-02-06 12:51:37,032] A new study created in memory with name: no-name-76be40bf-6865-482c-b480-e6fe7fd685d9


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-02-06 12:53:03,114] Trial 0 finished with value: 0.9531778414559412 and parameters: {'n_estimators': 562, 'max_depth': 10, 'learning_rate': 0.07587945476302646, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'reg_alpha': 0.15599452033620265, 'reg_lambda': 0.05808361216819946, 'min_child_weight': 9}. Best is trial 0 with value: 0.9531778414559412.
[I 2026-02-06 12:54:57,229] Trial 1 finished with value: 0.9545162668564713 and parameters: {'n_estimators': 721, 'max_depth': 8, 'learning_rate': 0.011852604486622221, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'reg_alpha': 0.21233911067827616, 'reg_lambda': 0.18182496720710062, 'min_child_weight': 2}. Best is trial 1 with value: 0.9545162668564713.
[I 2026-02-06 12:55:59,867] Trial 2 finished with value: 0.9549559079098083 and parameters: {'n_estimators': 513, 'max_depth': 7, 'learning_rate': 0.048875051677790424, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8447411578

In [29]:
xgb_best = XGBClassifier(**study.best_params, random_state=42)

ensemble = VotingClassifier(
    estimators=[
        ('xgb', xgb_best),
        ('lgbm', lgbm),
        ('gb', gb)
    ],
    voting='soft',
    weights=[2, 1, 1]
)

ensemble_scores = cross_val_score(ensemble, X_train, y_train, cv=cv, scoring='roc_auc')
print(f"Ensemble ROC-AUC: {ensemble_scores.mean():.5f} (+/- {ensemble_scores.std():.5f})")

ensemble.fit(X_train, y_train)
y_pred_proba = ensemble.predict_proba(X_test)[:, 1]
print(f"Validation ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.5f}")



Ensemble ROC-AUC: 0.95538 (+/- 0.00024)
Validation ROC-AUC: 0.95523


In [30]:
test_pred = ensemble.predict_proba(df_test)[:, 1]
submission = pd.DataFrame({'id': df_test_id, 'Heart Disease': test_pred})
submission.to_csv('submission.csv', index=False)

print('EOF')

EOF
